In [1]:
import os
os.environ["PYTHONUTF8"] = "1"

In [9]:
from transformers import BitsAndBytesConfig, AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset
import wandb
import torch

MODEL_NAME = "meta-llama/Llama-3.1-8B"
TRAINING_SET = "../data/SW_train_v1.jsonl"
CV_SET = "../data/SW_cv_v1.jsonl"
OUTPUT_DIR = "./output/suicidebot-v1"

In [3]:
import gc

if 'model' in dir():
    del model
if 'trainer' in dir():
    del trainer

gc.collect()
torch.cuda.empty_cache()

print(f"VRAM used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"VRAM free: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1e9:.2f} GB")

VRAM used: 0.00 GB
VRAM free: 17.18 GB


In [4]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
tokenizer.chat_template = "{% for message in messages %}{% if message['role'] == 'system' %}<|start_header_id|>system<|end_header_id|>\n\n{{ message['content'] }}<|eot_id|>{% elif message['role'] == 'user' %}<|start_header_id|>user<|end_header_id|>\n\n{{ message['content'] }}<|eot_id|>{% elif message['role'] == 'assistant' %}<|start_header_id|>assistant<|end_header_id|>\n\n{{ message['content'] }}<|eot_id|>{% endif %}{% endfor %}"

print(f"VRAM used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"VRAM free: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1e9:.2f} GB")

W0526 21:41:13.634000 19320 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
Loading weights:   1%|          | 2/291 [00:03<09:18,  1.93s/it]e:\Documents\Code\mentalhealth_ml\.venv-llm\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100%|██████████| 291/291 [01:02<00:00,  4.69it/s]


VRAM used: 5.71 GB
VRAM free: 11.47 GB


In [5]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

model.enable_input_require_grads()
model.config.use_cache = False
if hasattr(model, 'gradient_checkpointing_disable'):
    model.gradient_checkpointing_disable()

trainable params: 13,631,488 || all params: 8,043,892,736 || trainable%: 0.1695


In [10]:
dataset = load_dataset(
    "json",
    data_files={
        "train": TRAINING_SET,
        "cv": CV_SET
    }
)

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 47500
    })
    cv: Dataset({
        features: ['messages'],
        num_rows: 2500
    })
})


In [12]:
training_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    gradient_checkpointing=False,
    learning_rate=2e-5,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    bf16=True,
    fp16=False,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=3,
    load_best_model_at_end=True,
    report_to="wandb",
    max_length=256,
    completion_only_loss=True,
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [13]:
import wandb

wandb.finish()

wandb.init(
    project="suicidal-llm",
    name="llama3.1-8b-suicidewatch-v1",
    config={
        "model": MODEL_NAME,
        "rank": lora_config.r,
        "lora_alpha": lora_config.lora_alpha,
        "epochs": training_config.num_train_epochs,
        "batch_size": training_config.per_device_train_batch_size,
        "learning_rate": training_config.learning_rate,
    }
)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\moust\_netrc.
wandb: Currently logged in as: moustacherjr2351 (moustacherjr2351-na) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [14]:
trainer = SFTTrainer(
    model=model,
    args=training_config,
    train_dataset=dataset["train"],
    eval_dataset=dataset["cv"],
    processing_class=tokenizer,
)

In [15]:
# Loss diagnostic — run this before trainer.train() to confirm loss is real
dataloader = trainer.get_train_dataloader()
batch = next(iter(dataloader))
batch = {k: v.to("cuda") for k, v in batch.items()}

with torch.no_grad():
    outputs = model(**batch)

print(f"Loss: {outputs.loss.item()}")
print(f"VRAM used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"VRAM free: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1e9:.2f} GB")

Loss: 4.329759120941162
VRAM used: 5.88 GB
VRAM free: 11.30 GB


In [16]:
print("Starting training...")
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 128001}.


Starting training...


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
100,3.602298,3.612040,2.773993,163999.000000,0.382948


KeyboardInterrupt: 

In [ ]:
# Save adapter weights
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Adapter saved to {OUTPUT_DIR}")